# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**My lane: 2 - Refresh needed or not**<br>
Why? To identify which pages needs refresh, and i chose this because i want to learn how to do multi-class classification tasks


**The question:** <br>decision, action, cost of a wrong call
What decision does your work improve? Who acts on it? What does a wrong recommendation cost?

**Decision:**<br>
Flag pages which needs refresh
**Improvement:**<br>
This saves time of manual looking into pages and their content types. This saves time and resources to manually look at every page, and a human just look at flagged pages and check specific problem with that page.<br>
**Who acts?**<br>
A company/individual who wants to save time and effort to manually look at every page analytics.<br>
**Wrong recommendation costs**<br>
**False Positive:** We recommend a refresh for a page that was actually doing fine.<br> **Cost:** Wasted time/resources on unnecessary work. <br>**False Negative:** We miss a page that is dying. <br>
**Cost:** Permanent loss of traffic and revenue as Google stops indexing the outdated content.

**Careful words:** what I can and can't claim
Write what your work will be able to say (observed, directional, decision-support), and what it never will (causal proof, 'predicting Google').
<br>
My work is basically a decision-support. It will classify the decision and support in solution to the content optimization problems. Despite the model working, its still preferable to verify the decision by human involvement who are expert in this field.

**It will never:**<br>
1. Predict Google's core algorithm changes:I can only see how the site reacted and how should it react, not what Google will do next.
2. I can show that traffic dropped when content wasn't refreshed, but I can't prove it was the only reason.
3. Guarantee Rankings: I can suggest improvements, but I cannot predict or control specific ranking positions.
4. Auto refresh the page contents, it will only provide a decision whether to refresh or not.
5. Fix SEO issues and its errors
6. Identify which user visits your page
7. Claim that decision made by model is 100% certain because model trained on data which is of a limited time period not whole year.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Unit of analysis + time window**<br>
One row = one what, over which dates? State it, then verify it below.

I made a 90 days time window selected from latest 3 months from date 2026-04-01 to 2026-06-30.

One row = One client's one reported content page needed refresh or not.

Trained on 60day window (2026-04-01 to 2026-05-31), tested on June (month=06) 30day window (2026-06-01 to 2026-06-30).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

1.3: Get 3 months latest window

In [ ]:
clients_last_3m = con.sql(f"""
    SELECT report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-05-31'
""")

1.4: 90 day window start and end dates/ Time window

In [ ]:
con.sql("""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM clients_last_3m
""").show()

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Features:**<br>
'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used', 'model_used', ''

**Proxy label:** 'decline_score'

**Context:**<br>
'report_date', 'client_hash_id', 'content_hash_id', 'url_hash_id', 'client_has_gsc', 'keyword_hash_id', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'content_created_date', 'content_updated_date', 'last_optimized_date','keyword_created_date', 'is_published', 'is_deleted',

**Excluded:**
1. month - report_date already shows month so no need
2. optimization_eligible_date - leakage for output refresh_needed so dropped
3. char_count - Not matter for declining signal
4. word_count - Not matter for declining signal.
5. category_count - Not matter for declining signal.
6. keyword_char_count - Not matter for declining signal
7. keyword_token_count - Not matter for declining signal.
8. url_char_count - Not matter for declining signal

**Weak picks + leakage check** <br>
Which picks look wrong and why? Confirm no product flags or future windows leaked in.

**Weak picks:**<br>
1. Low-volume ratio instability:<br> rows tied to content_8fcb62f89b4e1003 and content_c1ca33bd9c246c60 sit at 20-100 gsc_impressions_last30d with 0-1 gsc_clicks_last30d. A ctr_last30 built on that few impressions moves drastically on one click, unlike rows such as content_904b8c2dc4638c4d (900+ impressions), where the same ratio is far more stable, yet both get treated as equally reliable evidence if the scoring condition is a flat threshold check.<br>
2. Directionally mixed row inside a repeated content_id:<br> the client-3821400 snapshot of content_904b8c2dc4638c4d shows ga4_sessions_last30/sessions_organic_last30 higher than its own sibling snapshots (5 vs 1, 3 vs 0), an improving signal sitting inside a set of rows that otherwise decline in lockstep, all scored identically at 17.<br>
3. NaN-bearing rows:<br> backlinks_prev30/last30 is in 7 of the 20 rows — any condition reading this column directly gets an undetermined input for those specific rows, not a true zero or true decline.

**Leakage check:**<br>
Scanning every column in the sample, all are prev30 or last30 window aggregates (GSC/GA4 metrics) or static dim_content fields. Nothing extends beyond the last30 window, and no explicit product/QA flag column exists in this feature set. No future-window or flag leakage found in the columns shown.

In [ ]:
clients_last_3m = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-05-30'
      AND client_has_gsc == 'true'
      AND client_has_ga4 == 'true'
      AND gsc_data_available == 'true'
      AND ga4_data_available == 'true'
""").df()

In [ ]:
dimf_content = con.sql(f"""
    SELECT *
    FROM {TABLES['dim_content']}
""").df()

In [ ]:
combined_data = clients_last_3m.merge(
    dimf_content,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
    )

In [ ]:
combined_data = combined_data[combined_data["is_published"] == True]

In [ ]:
combined_data = combined_data[combined_data["is_deleted"] == False]

In [ ]:
combined_data_v2 = combined_data.drop(columns=['client_has_gsc','client_has_ga4', 'gsc_data_available','gsc_sum_position', 'ga4_data_available','ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other','keyword_hash_id',
       'url_hash_id', 'keyword_char_count', 'keyword_token_count',
       'url_char_count','content_type', 'search_volume', 'competition', 'competition_level','main_intent', 'category_count',
       'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'month',
       'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'])

In [ ]:
combined_data_v2["report_date"] = pd.to_datetime(combined_data_v2["report_date"])

training_window_01 = combined_data_v2[
    (combined_data_v2["report_date"] >= "2026-03-01") & (combined_data_v2["report_date"] <= "2026-03-31")
]
training_window_02 = combined_data_v2[
    (combined_data_v2["report_date"] >= "2026-04-01") & (combined_data_v2["report_date"] <= "2026-04-30")
]
testing_window = combined_data_v2[
    (combined_data_v2["report_date"] >= "2026-05-01") & (combined_data_v2["report_date"] <= "2026-05-31")
]

In [ ]:
print(training_window_01['report_date'].min())
print(training_window_01['report_date'].max())
print(len(training_window_01))
print(training_window_01.columns)

In [ ]:
reference_date = dimf_content["content_updated_date"].max()
training_window_01["ctr"] = training_window_01["gsc_clicks"] / training_window_01["gsc_impressions"]
training_window_01["engagement_rate"] = training_window_01["ga4_engaged_sessions"] / training_window_01["ga4_sessions"]
training_window_01["scroll_events_rate"] = training_window_01["scroll_events"] / training_window_01["ga4_pageviews"].replace(0, np.nan)
training_window_01["days_since_update"] = (reference_date - training_window_01["content_updated_date"]).dt.days

In [ ]:
training_window_01 = training_window_01.rename(columns={"gsc_impressions": "gsc_impressions_prev30d", "gsc_clicks": "gsc_clicks_prev30d",
                                                        "gsc_avg_position":"gsc_avg_position_prev30d","ga4_pageviews":"ga4_pageviews_prev30d",
                                                        "ga4_sessions":"ga4_sessions_prev30d", "ga4_users":"ga4_users_prev30d",
                                                        "ga4_engaged_sessions":"'ga4_engaged_session_prev30", "ga4_total_engagement_sec":"ga4_total_engagement_sec_prev30",
                                                        "sessions_organic":"sessions_organic_prev30", "sessions_direct":"sessions_direct_prev30",
                                                        "sessions_referral":"sessions_referral_prev30", "sessions_social":"sessions_social_prev30",
                                                        "sessions_paid":"sessions_paid_prev30","sessions_ai":"sessions_ai_prev30",
                                                        "scroll_events":"scroll_events_prev30","cpc":"cpc_prev30", "backlinks":"backlinks_prev30",
                                                        "ctr":"ctr_prev30", "engagement_rate":"engagement_rate_prev30", "scroll_events_rate":"scroll_events_rate_prev30",
                                                        "days_since_update":"days_since_update_prev30"})
print(training_window_01.columns)

In [ ]:
print(training_window_02['report_date'].min())
print(training_window_02['report_date'].max())
print(len(training_window_02))
print(training_window_02.columns)

In [ ]:
reference_date = dimf_content["content_updated_date"].max()
training_window_02["ctr"] = training_window_02["gsc_clicks"] / training_window_02["gsc_impressions"]
training_window_02["engagement_rate"] = training_window_02["ga4_engaged_sessions"] / training_window_02["ga4_sessions"]
training_window_02["scroll_events_rate"] = training_window_02["scroll_events"] / training_window_02["ga4_pageviews"].replace(0, np.nan)
training_window_02["days_since_update"] = (reference_date - training_window_02["content_updated_date"]).dt.days

In [ ]:
training_window_02 = training_window_02.rename(columns={"gsc_impressions": "gsc_impressions_last30d", "gsc_clicks": "gsc_clicks_last30d",
                                                        "gsc_avg_position":"gsc_avg_position_last30d","ga4_pageviews":"ga4_pageviews_last30d",
                                                        "ga4_sessions":"ga4_sessions_last30d", "ga4_users":"ga4_users_last30d",
                                                        "ga4_engaged_sessions":"'ga4_engaged_session_last30", "ga4_total_engagement_sec":"ga4_total_engagement_sec_last30",
                                                        "sessions_organic":"sessions_organic_last30", "sessions_direct":"sessions_direct_last30",
                                                        "sessions_referral":"sessions_referral_last30", "sessions_social":"sessions_social_last30",
                                                        "sessions_paid":"sessions_paid_last30","sessions_ai":"sessions_ai_last30",
                                                        "scroll_events":"scroll_events_last30","cpc":"cpc_last30", "backlinks":"backlinks_last30",
                                                        "ctr":"ctr_last30", "engagement_rate":"engagement_rate_last30", "scroll_events_rate":"scroll_events_rate_last30",
                                                        "days_since_update":"days_since_update_last30"})
print(training_window_02.columns)

In [ ]:
features_column = training_window_01.merge(
    training_window_02,
    on=("client_hash_id", "content_hash_id", "content_created_date", "content_updated_date"),
    how="inner"
)

In [ ]:
features_column["decline_score"] = (
    (features_column["ctr_last30"] < features_column["ctr_prev30"]).astype(int) +
    (features_column["engagement_rate_last30"] <= features_column["engagement_rate_prev30"]).astype(int) +
    (features_column["scroll_events_rate_last30"] <= features_column["scroll_events_rate_prev30"]).astype(int) +
    (features_column["gsc_avg_position_last30d"] >= features_column["gsc_avg_position_prev30d"]).astype(int) +
    (features_column["gsc_impressions_last30d"] <= features_column["gsc_impressions_prev30d"]).astype(int) +
    (features_column["gsc_clicks_last30d"] <= features_column["gsc_clicks_prev30d"]).astype(int) +
    (features_column["ga4_pageviews_last30d"] <= features_column["ga4_pageviews_prev30d"]).astype(int) +
    (features_column["ga4_users_last30d"] <= features_column["ga4_users_prev30d"]).astype(int)+
    (features_column["ga4_total_engagement_sec_last30"] <= features_column["ga4_total_engagement_sec_prev30"]).astype(int) +
    (features_column["sessions_organic_last30"] <= features_column["sessions_organic_prev30"]).astype(int) +
    (features_column["sessions_direct_last30"] <= features_column["sessions_direct_prev30"]).astype(int) +
    (features_column["sessions_referral_last30"] <= features_column["sessions_referral_prev30"]).astype(int) +
    (features_column["sessions_social_last30"] <= features_column["sessions_social_prev30"]).astype(int) +
    (features_column["sessions_paid_last30"] <= features_column["sessions_paid_prev30"]).astype(int) +
    (features_column["sessions_ai_last30"] <= features_column["sessions_ai_prev30"]).astype(int) +
    (features_column["scroll_events_last30"] <= features_column["scroll_events_prev30"]).astype(int) +
    (features_column["cpc_last30"] <= features_column["cpc_prev30"]).astype(int)
)

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
data = pd.read_csv("data_for_baseline_action_score.csv")

In [ ]:

data = data.dropna(subset=['cpc_prev30']).reset_index(drop=True)

In [ ]:
data = data.dropna(subset=['backlinks_prev30']).reset_index(drop=True)

In [ ]:

data = data.dropna(subset=['engagement_rate_prev30']).reset_index(drop=True)

In [ ]:

data = data.dropna(subset=['scroll_events_rate_prev30']).reset_index(drop=True)

In [ ]:

data = data.dropna(subset=['engagement_rate_last30']).reset_index(drop=True)

In [ ]:
data = data.dropna(subset=['scroll_events_rate_last30']).reset_index(drop=True)

In [ ]:

# Set random seed for reproducibility
np.random.seed(42)

# Get unique clients
unique_clients = data['client_hash_id'].unique()

# Shuffle and split client IDs (80/20)
np.random.shuffle(unique_clients)
split_idx = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:split_idx]
test_clients = unique_clients[split_idx:]

# Split the dataset based on client groups
train_data = data[data['client_hash_id'].isin(train_clients)].copy()
test_data = data[data['client_hash_id'].isin(test_clients)].copy()

print(f"Total unique clients: {len(unique_clients)}")
print(f"Training clients: {len(train_clients)} | Rows: {len(train_data)}")
print(f"Testing clients: {len(test_clients)} | Rows: {len(test_data)}")

# Verify that there is no client overlap
overlap = set(train_data['client_hash_id']).intersection(set(test_data['client_hash_id']))
print(f"Client overlap between train and test: {len(overlap)}")

In [ ]:
X_train = train_data.drop(columns=['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'decline_score'], axis=1)
y_train = train_data['decline_score']

In [ ]:
X_test = test_data.drop(columns=['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'decline_score'], axis=1)
y_test = test_data['decline_score']

In [ ]:
model = RandomForestClassifier(n_estimators = 10,random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba, multi_class='ovo', labels=model.classes_)

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

In [ ]:
print("Actual (y_test) vs Predicted (y_pred):")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(comparison_df.head(20))
display(comparison_df.tail(20))
print(f"Total rows: {len(comparison_df)}\n")

## 5. Limitations

*What this work cannot claim.*

Analysis of Overlap and Score Drop

The Leakage Mechanism: Under a standard random split, multiple time-series snapshots of the exact same content items (content_hash_id) are split arbitrarily. As a result, approximately 97.38% of the content items in the test set already exist in the training set with highly similar historic features.
The Consequence: This severe leak inflates the model's performance to an unrealistic 0.5821 F1-score. When we transition to an honest grouped split (grouping strictly by content_hash_id or client_hash_id so that unseen items are tested), the model can no longer rely on memorizing specific content behaviors, which drops the F1-score to an honest 0.444.

In [ ]:
train_content = set(X_train.index.map(lambda idx: data.loc[idx, 'content_hash_id']))
test_content = set(X_test.index.map(lambda idx: data.loc[idx, 'content_hash_id']))

overlap = train_content.intersection(test_content)
overlap_pct = (len(overlap) / len(test_content)) * 100

print(f"Unique content IDs in Train: {len(train_content):,}")
print(f"Unique content IDs in Test: {len(test_content):,}")
print(f"Content IDs overlapping between Train and Test: {len(overlap):,}")
print(f"Overlap Percentage (Test content also in Train): {overlap_pct:.2f}%")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The queue is ordered by how strongly the available signals agree that a page has declined, not by predicted content strategy. At the extremes of the score range, the model's evaluated accuracy is highest (F1 ~0.75), so a human can act on those rankings with more confidence. In the middle of the score range, the same score can arise from different combinations of reason codes, and model accuracy is weaker (F1 ~0.42-0.58) — so "what to do first" there means "look here next, but verify manually before treating the ranking as decisive," rather than treating the order as a strict priority list.

Reason codes
IF gsc_clicks_last30 ≤ gsc_clicks_prev30 OR IF gsc_impressions_last30 ≤ gsc_impressions_prev30 OR IF gsc_avg_position_last30 ≥ gsc_avg_position_prev30 OR IF ctr_last30 < ctr_prev30 OR IF engagement_rate_last30 ≤ engagement_rate_prev30 OR IF scroll_events_rate_last30 ≤ scroll_events_rate_prev30 OR IF ga4_pageviews_last30 ≤ ga4_pageviews_prev30 OR IF ga4_users_last30 ≤ ga4_users_prev30 OR IF ga4_total_engagement_sec_last30 ≤ ga4_total_engagement_sec_prev30 OR IF sessions_organic_last30 ≤ sessions_organic_prev30 OR IF sessions_direct_last30 ≤ sessions_direct_prev30 OR IF sessions_referral_last30 ≤ sessions_referral_prev30 OR IF sessions_social_last30 ≤ sessions_social_prev30 OR IF sessions_paid_last30 ≤ sessions_paid_prev30 OR IF sessions_ai_last30 ≤ sessions_ai_prev30 OR IF scroll_events_last30 ≤ scroll_events_prev30 OR IF cpc_last30 ≤ cpc_prev30

In [ ]:
ranked_queue = data.sort_values('decline_score', ascending=False)
ranked_queue[:20]

In [ ]:
top_queue = ranked_queue.head(20)

summary_stats = top_queue[[
    'decline_score',
    'ga4_pageviews_prev30d',
    'ga4_pageviews_last30d',
    'gsc_impressions_prev30d',
    'gsc_impressions_last30d',
    'gsc_clicks_prev30d',
    'gsc_clicks_last30d',
    'gsc_avg_position_prev30d',
    'gsc_avg_position_last30d',
    'ga4_sessions_prev30d',
    'ga4_sessions_last30d',
    'ga4_users_prev30d',
    'ga4_users_last30d',
    "'ga4_engaged_session_prev30",
    "'ga4_engaged_session_last30",
    'ga4_total_engagement_sec_prev30',
    'ga4_total_engagement_sec_last30',
    'sessions_organic_prev30',
    'sessions_organic_last30',
    'sessions_direct_prev30',
    'sessions_direct_last30',
    'sessions_referral_prev30',
    'sessions_referral_last30',
    'sessions_social_prev30',
    'sessions_social_last30',
    'sessions_paid_prev30',
    'sessions_paid_last30',
    'sessions_ai_prev30',
    'sessions_ai_last30',
    'scroll_events_prev30',
    'scroll_events_last30',
    'cpc_prev30',
    'cpc_last30',
    'backlinks_prev30',
    'backlinks_last30',
    'ctr_prev30',
    'ctr_last30',
    'engagement_rate_prev30',
    'engagement_rate_last30',
    'scroll_events_rate_prev30',
    'scroll_events_rate_last30',
    'days_since_update_last30'
]].describe()

display(summary_stats)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
metric_pairs = [
    ('ga4_pageviews_prev30d', 'ga4_pageviews_last30d', 'Pageviews'),
    ('gsc_impressions_prev30d', 'gsc_impressions_last30d', 'Impressions'),
    ('gsc_clicks_prev30d', 'gsc_clicks_last30d', 'Clicks'),
    ('gsc_avg_position_prev30d', 'gsc_avg_position_last30d', 'Avg Position'),
    ('ga4_sessions_prev30d', 'ga4_sessions_last30d', 'Sessions'),
    ('ga4_users_prev30d', 'ga4_users_last30d', 'Users'),
    ("'ga4_engaged_session_prev30", "'ga4_engaged_session_last30", 'Engaged Sessions'),
    ('ga4_total_engagement_sec_prev30', 'ga4_total_engagement_sec_last30', 'Engagement (sec)'),
    ('sessions_organic_prev30', 'sessions_organic_last30', 'Organic Sessions'),
    ('sessions_direct_prev30', 'sessions_direct_last30', 'Direct Sessions'),
    ('sessions_referral_prev30', 'sessions_referral_last30', 'Referral Sessions'),
    ('sessions_social_prev30', 'sessions_social_last30', 'Social Sessions'),
    ('sessions_paid_prev30', 'sessions_paid_last30', 'Paid Sessions'),
    ('sessions_ai_prev30', 'sessions_ai_last30', 'AI Sessions'),
    ('scroll_events_prev30', 'scroll_events_last30', 'Scroll Events'),
    ('cpc_prev30', 'cpc_last30', 'CPC'),
    ('backlinks_prev30', 'backlinks_last30', 'Backlinks'),
    ('ctr_prev30', 'ctr_last30', 'CTR'),
    ('engagement_rate_prev30', 'engagement_rate_last30', 'Engagement Rate'),
    ('scroll_events_rate_prev30', 'scroll_events_rate_last30', 'Scroll Rate'),
]

n_metrics = len(metric_pairs)
n_cols = 4
n_rows = -(-n_metrics // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for i, (prev_col, last_col, label) in enumerate(metric_pairs):
    ax = axes[i]
    prev_mean = top_queue[prev_col].mean()
    last_mean = top_queue[last_col].mean()

    bars = ax.bar(['prev30', 'last30'], [prev_mean, last_mean],
                   color=['#4C72B0', '#C44E52'], width=0.5)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=7)

    for bar, val in zip(bars, [prev_mean, last_mean]):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2g}',
                ha='center', va='bottom', fontsize=7)

# hide unused subplots
for j in range(n_metrics, len(axes)):
    axes[j].axis('off')

fig.suptitle('Top 20 Decline-Score Pages: prev30 vs. last30 Mean by Metric', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('work/outputs/top20_prev_vs_last_grid.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis('off')

boxes = [
    (0.3, 1, 2.2, 1, '#4C72B0', "decline_score\n(0–17)\nproxy label"),
    (2.9, 1, 2.2, 1, '#DD8452', "Reason codes\n(which signals\ntriggered)"),
    (5.5, 1, 2.2, 1, '#55A868', "Human SEO\nreviewer"),
    (8.1, 1, 1.6, 1, '#C44E52', "Action\n(effort level,\nnot automated)"),
]

for x, y, w, h, color, label in boxes:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                          linewidth=1.5, edgecolor='black', facecolor=color, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=10, color='white', fontweight='bold')

arrow_positions = [(2.5, 1.5, 2.9, 1.5), (5.1, 1.5, 5.5, 1.5), (7.7, 1.5, 8.1, 1.5)]
for x1, y1, x2, y2 in arrow_positions:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.text(5, 2.6, 'Decision-support pipeline: ranking informs priority, human decides the fix',
        ha='center', fontsize=11, style='italic', color='#333333')
ax.text(5, 0.4, 'No automated content changes occur at any step — output is advisory only',
        ha='center', fontsize=9.5, color='#555555')

plt.tight_layout()
plt.savefig('work/outputs/decision_support_workflow.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
classes = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
f1_scores = [0.00, 0.00, 0.22, 0.46, 0.48, 0.39, 0.43, 0.45, 0.54, 0.51, 0.55, 0.55, 0.65, 0.75, 0.70]
support = [2, 17, 528, 2444, 4380, 4165, 4576, 7599, 11618, 13127, 14309, 15054, 23393, 30413, 14836]

fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.bar(classes, f1_scores, color='#1f77b4', alpha=0.85, width=0.6, label='F1-score')
ax1.set_xlabel('decline_score class', fontsize=12)
ax1.set_ylabel('F1-score', color='#1f77b4', fontsize=12)
ax1.set_ylim(0, 1.0)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_xticks(classes)
ax1.axhline(0.44, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Macro F1 (0.44)')

ax2 = ax1.twinx()
ax2.plot(classes, support, color='#d62728', marker='o', linewidth=2, markersize=5, label='Support (n)')
ax2.set_ylabel('Support (test set n)', color='#d62728', fontsize=12)
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='#d62728')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.title('Model Reliability Varies Sharply by Class (F1-score vs. Sample Support)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig('work/outputs/f1_by_class_support.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Figure 1: Confidence tiers by class — for "What person must check?" ---

classes = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
f1_scores = [0.00, 0.00, 0.22, 0.46, 0.48, 0.39, 0.43, 0.45, 0.54, 0.51, 0.55, 0.55, 0.65, 0.75, 0.70]

confidence_tier = []
colors = []
for c, f1 in zip(classes, f1_scores):
    if c in [3, 4]:
        confidence_tier.append('Near-zero data')
        colors.append('#C44E52')
    elif 8 <= c <= 13:
        confidence_tier.append('Weakest reliability')
        colors.append('#DD8452')
    else:
        confidence_tier.append('Higher confidence')
        colors.append('#55A868')

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(classes, f1_scores, color=colors, width=0.6, edgecolor='black', linewidth=0.5)

ax.set_xlabel('decline_score class', fontsize=12)
ax.set_ylabel('F1-score', fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_xticks(classes)
ax.set_title('Score Confidence by Class — Where Extra Skepticism Is Required', fontsize=13, fontweight='bold')

legend_elements = [
    plt.Rectangle((0,0),1,1, color='#C44E52', label='Classes 3–4: near-zero training data'),
    plt.Rectangle((0,0),1,1, color='#DD8452', label='Classes 8–13: weakest model reliability'),
    plt.Rectangle((0,0),1,1, color='#55A868', label='Other classes: higher confidence'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('work/outputs/confidence_tiers_by_class.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis('off')

steps = [
    (0.3, 1, 2.3, 1, '#4C72B0', "Model output:\ndecline_score"),
    (3.0, 1, 2.3, 1, '#C44E52', "Diagnose\nspecific problem\n(HUMAN ONLY)"),
    (5.7, 1, 2.3, 1, '#C44E52', "Decide &\ntrigger refresh\n(HUMAN ONLY)"),
]

for x, y, w, h, color, label in steps:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                          linewidth=1.5, edgecolor='black', facecolor=color, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=10, color='white', fontweight='bold')

ax.annotate('', xy=(3.0, 1.5), xytext=(2.6, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.annotate('', xy=(5.7, 1.5), xytext=(5.3, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.annotate('NO AUTOMATED PATH', xy=(8.3, 1.5), xytext=(8.6, 2.4),
            fontsize=10, color='#C44E52', fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='-[', lw=2, color='#C44E52'))

ax.text(5, 0.3, 'The model stops at the score — diagnosis and action are never automated',
        ha='center', fontsize=10, style='italic', color='#333333')

plt.tight_layout()
plt.savefig('work/outputs/never_automated_steps.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')

ax.text(5, 6.5, 'Staleness Triggers for the Recommendation Queue',
        ha='center', fontsize=14, fontweight='bold')

triggers = [
    (1, '30-day window\nexpiry',
     'More than 30 days since\nlast queue regeneration',
     'MEASURABLE\nTODAY', '#55A868'),
    (2, 'External shock /\nseasonality',
     'Algorithm update or seasonal\nshift invalidates baseline',
     'NEEDS SETUP\n(control-page check)', '#DD8452'),
    (3, 'Rising human\nfalse-positive rate',
     'Reviewers flag top-ranked\npages as not needing action',
     'NEEDS SETUP\n(feedback log)', '#DD8452'),
]

x_positions = [0.6, 3.9, 7.2]
width = 2.7

for (num, title, desc, status, color), x in zip(triggers, x_positions):
    # main box
    box = FancyBboxPatch((x, 2.6), width, 2.0, boxstyle="round,pad=0.08",
                          linewidth=1.5, edgecolor='black', facecolor='#EAEAF2', alpha=0.95)
    ax.add_patch(box)
    ax.text(x + width/2, 4.15, f'{num}. {title}', ha='center', va='center',
            fontsize=11, fontweight='bold')
    ax.text(x + width/2, 3.3, desc, ha='center', va='center', fontsize=9.2, color='#333333')

    # status badge
    badge = FancyBboxPatch((x + 0.15, 1.5), width - 0.3, 0.8, boxstyle="round,pad=0.05",
                            linewidth=1.2, edgecolor='black', facecolor=color, alpha=0.9)
    ax.add_patch(badge)
    ax.text(x + width/2, 1.9, status, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

    # connector line
    ax.plot([x + width/2, x + width/2], [2.6, 2.3], color='black', linewidth=1)

ax.text(5, 0.6, 'Trigger (1) is structural and measurable from existing data now.\n'
                'Triggers (2) and (3) require new monitoring processes before they can be tracked.',
        ha='center', fontsize=10, style='italic', color='#444444')

plt.tight_layout()
plt.savefig('work/outputs/staleness_triggers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ranked_queue[:20].to_csv('work/outputs/ranked_queue.csv', index=False)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
